# Evaluation of Agent Systems

This notebook turns the project from a demo into a measurable experiment. We compare the baseline and the agent workflow on the same dataset and interpret what the metrics actually mean.

## Learning goals

- Understand why evaluation matters for agent systems.
- Inspect the structure of a small evaluation dataset.
- Compute correctness, retrieval, grounding, abstention, latency, and step-count metrics.
- Compare baseline behavior against the agentic workflow in a reproducible way.


## Concept explanation

We begin with the same environment check because evaluation should be reproducible. If the interpreter is wrong, the metric story is unreliable from the start.


In [ ]:
import sys
print(sys.executable)


This setup cell imports the evaluation helpers from `src/` and normalizes the notebook working directory. Everything measured below comes from the same code paths used by the tests.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.evaluator import load_eval_dataset, run_evaluation_suite

pd.set_option('display.max_colwidth', 140)


### Why evaluation matters

A workflow can look impressive on one cherry-picked question and still behave badly overall. Evaluation matters because it tests the full loop repeatedly and surfaces whether improvements are real or only anecdotal.

### Designing evaluation datasets

This dataset mixes straightforward lookups, comparisons, summaries, multi-hop questions, and insufficient-evidence cases. That variety lets us see where agentic orchestration helps and where it still fails.

## Implementation


In [ ]:
dataset = load_eval_dataset()
dataset_frame = pd.DataFrame(dataset)
dataset_frame[['id', 'question', 'question_type', 'expected_status']].head(10)


### Metrics

We now run both systems across repeated trials. The built-in metrics cover answer correctness, retrieval hit rate, grounding score, abstain precision, latency, and the number of reasoning steps recorded in the trace.


In [ ]:
results, summary = run_evaluation_suite(repeats=2, persist_outputs=True)
summary


The raw evaluation frame is useful because it shows per-question behavior instead of only averages. Looking at the columns helps you understand which signals drive a good or bad result.


In [ ]:
results[['system', 'question_id', 'question', 'answer_correctness', 'retrieval_hit_rate', 'grounding_pass', 'abstained', 'latency_seconds', 'average_steps', 'failure_type']].head(12)


## Experiment

A helpful experiment is to compare metrics by system side by side, then break them down by question type. This shows whether the agent workflow helps everywhere or mostly on certain classes of problems.


In [ ]:
comparison = summary.set_index('system')
question_type_breakdown = (
    results.groupby(['system', 'expected_question_type'])[['answer_correctness', 'retrieval_hit_rate', 'grounding_pass_rate', 'latency_seconds', 'average_steps']]
    .mean()
    .round(3)
)
comparison, question_type_breakdown


## Result analysis

The comparison table is where you should look for trade-offs. A stronger workflow may take more steps and slightly more latency, but that cost is often justified if it improves grounding and abstention quality.


In [ ]:
metric_deltas = summary.set_index('system').diff().dropna()
metric_deltas.index = ['agent_minus_baseline']
metric_deltas


## Takeaways

- Evaluation turns architecture claims into measurable evidence.
- Baseline RAG is a useful control, not an enemy.
- Stronger grounding and abstention are often worth modest extra latency.
- The next step is to inspect failures, not just averages.
